Broze Layer - Ingestion

In [0]:
%run ./00_config

In [0]:

import requests
import json
from datetime import datetime, timezone

run_id = new_run_id()
ingest_timestamp = datetime.now(timezone.utc).isoformat() #We'll store this inside the data as metadata (different purpose than the folder name — this one travels with the actual rows).

print(f"Starting ingest run: {run_id}")

In [0]:
response = requests.get(API_BASE_URL, params=API_QUERY_PARAMS)
response.raise_for_status() #fail loudly" principle in action - if the server respond with an error code, stops the notebook with clear Python exception

raw_json = response.json()

In [0]:
print(json.dumps(raw_json, indent=2)[:500])  # print first 500 characters, pretty-printed

In [0]:
# save this to disk, into a run-specific folder
run_folder = f"{BRONZE_PATH}/{run_id}"

dbutils.fs.mkdirs(run_folder)

print(f"Created bronze run folder: {run_folder}")

In [0]:
bronze_record = {
    "ingest_run_id": run_id,
    "ingest_timestamp": ingest_timestamp,
    "source_url": API_BASE_URL,
    "raw_response": raw_json,
}

output_path = f"{run_folder}/raw_response.json"

with open(output_path, "w") as f:
    json.dump(bronze_record, f, indent=2)

print(f"Wrote raw bronze data to: {output_path}")

In [0]:
%sql
--SHOW CATALOGS
--SHOW VOLUMES

In [0]:
features = raw_json.get("features", [])
rows_in = len(features)

print(f"Run ID: {run_id}")
print(f"Rows ingested from API: {rows_in}")